In [20]:
!pip install -q groq

In [21]:
import os
import json
from groq import Groq
from getpass import getpass
os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ_API_KEY:")
client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "openai/gpt-oss-120b"

Enter your GROQ_API_KEY:··········


In [22]:
def get_customer(customer_id):
    customers = {
        "C101": {
            "name": "Rahul",
            "plan": "Premium"
        },
        "C102": {
            "name": "Priya",
            "plan": "Basic"
        }
    }
    return customers.get(customer_id, "Customer not found")

In [23]:
def get_order(order_id):
    orders = {
        "ORD1001": {
            "customer_id": "C101",
            "status": "Shipped",
            "amount": 2500
        },
        "ORD1002": {
            "customer_id": "C102",
            "status": "Delivered",
            "amount": 1800
        }
    }

    return orders.get(order_id, "Order not found")

def calculate_refund(amount, percentage):
    return amount * percentage / 100

In [24]:
available_tools = {
    "get_customer": get_customer,
    "get_order": get_order,
    "calculate_refund": calculate_refund
}

In [25]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_customer",
            "description": "Get customer information using a customer ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "customer_id": {
                        "type": "string",
                        "description": "The customer ID, for example C101 or C102."
                    }
                },
                "required": ["customer_id"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "get_order",
            "description": "Get order information using an order ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "The order ID, for example ORD1001 or ORD1002."
                    }
                },
                "required": ["order_id"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "calculate_refund",
            "description": "Calculate a refund amount from an original amount and refund percentage.",
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {
                        "type": "number",
                        "description": "Original transaction amount."
                    },
                    "percentage": {
                        "type": "number",
                        "description": "Refund percentage."
                    }
                },
                "required": ["amount", "percentage"]
            }
        }
    }
]


In [28]:
def run_agent(user_request, max_iterations=5):

    messages = [
        {
            "role": "system",
            "content": """
            You are an operations assistant. You have access to tools.
            Use the tools whenever the user asks for customer, order, or refund information.

            Never invent information.

            Continue using tools until the user's question has been completely answered."""
        },
        {
            "role": "user",
            "content": user_request
        }
    ]

    for iteration in range(max_iterations):

        print(f"\n========== ITERATION {iteration + 1} ==========")

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        assistant_message = response.choices[0].message

        messages.append(assistant_message)

        # --------------------------------
        # Agent has finished
        # --------------------------------

        if not assistant_message.tool_calls:

            print("Agent decided it is finished.")

            return assistant_message.content

        # --------------------------------
        # Execute requested tools
        # --------------------------------

        for tool_call in assistant_message.tool_calls:

            name = tool_call.function.name

            arguments = json.loads(tool_call.function.arguments)

            print(f"Tool requested: {name}")
            print(f"Arguments: {arguments}")

            if name not in available_tools:
                result = {
                    "error": f"Unknown tool: {name}"
                }

            else:
                function = available_tools[name]

                result = function(**arguments)

            print(f"Tool result: {result}")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": name,
                "content": json.dumps(result)
            })

    return "Agent stopped because maximum iterations were reached."


In [29]:
answer = run_agent(
    """
    Find customer C102,
    check their order ORD1002,
    and tell me the order status.
    """
)

print("\nFINAL ANSWER:")
print(answer)


========== ITERATION 1 ==========
Tool requested: get_customer
Arguments: {'customer_id': 'C102'}
Tool result: {'name': 'Priya', 'plan': 'Basic'}

========== ITERATION 2 ==========
Tool requested: get_order
Arguments: {'order_id': 'ORD1002'}
Tool result: {'customer_id': 'C102', 'status': 'Delivered', 'amount': 1800}

========== ITERATION 3 ==========
Agent decided it is finished.

FINAL ANSWER:
The order **ORD1002** for customer **C102 (Priya)** has a status of **Delivered**.


In [30]:
answer = run_agent(
    """
    Please investigate customer C102.
    I want to know who the customer is, what plan they have,
    whether their order ORD1002 has been delivered,
    and if it has been delivered, calculate a 20% refund
    on the order amount.
    """
)

print(answer)


========== ITERATION 1 ==========
Tool requested: get_customer
Arguments: {'customer_id': 'C102'}
Tool result: {'name': 'Priya', 'plan': 'Basic'}

========== ITERATION 2 ==========
Tool requested: get_order
Arguments: {'order_id': 'ORD1002'}
Tool result: {'customer_id': 'C102', 'status': 'Delivered', 'amount': 1800}

========== ITERATION 3 ==========
Tool requested: calculate_refund
Arguments: {'amount': 1800, 'percentage': 20}
Tool result: 360.0

========== ITERATION 4 ==========
Agent decided it is finished.
**Customer Details (C102)**  
- **Name:** Priya  
- **Plan:** Basic  

**Order Details (ORD1002)**  
- **Customer ID:** C102  
- **Status:** Delivered  
- **Order Amount:** $1,800.00  

**Refund Calculation**  
- Since the order has been delivered, a 20 % refund is applicable.  
- **Refund Amount:** $360.00  

Let me know if you need anything else (e.g., processing the refund, checking other orders, or updating the plan).
